# Petition EDA

Exploratory analysis of the Change.org data-center petition dataset in `dc_petitions/petitioner.db`.

The database is opened **read-only**, so this notebook is safe to run even while a collection is still writing to it (SQLite WAL allows concurrent readers). To refresh after a collection finishes, just re-run all cells.

Each petition carries one or more provenance labels in `petition_source` (`DCDB`, `change.org`, `fullquery`).

In [ ]:
import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl

# Locate the database whether the notebook runs from the repo root or a subfolder.
DB_PATH = None
for _p in [Path.cwd(), *Path.cwd().parents]:
    _cand = _p / "dc_petitions" / "petitioner.db"
    if _cand.exists():
        DB_PATH = _cand
        break
if DB_PATH is None:
    raise FileNotFoundError("could not locate dc_petitions/petitioner.db")

# Read-only URI connection: never blocks the writer during a live collection.
conn = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)


def q(sql: str, params: tuple = ()) -> pl.DataFrame:
    """Run a read-only query and return a polars DataFrame."""
    cur = conn.execute(sql, params)
    cols = [d[0] for d in cur.description]
    return pl.DataFrame(cur.fetchall(), schema=cols, orient="row")


plt.rcParams.update(
    {"figure.figsize": (9, 4), "axes.grid": True, "grid.alpha": 0.3}
)
print(f"connected: {DB_PATH}")

## 1. Overview and provenance

In [ ]:
tables = ["petition", "comment", "decision_maker", "tag", "observation",
          "run", "petition_source"]
counts = {t: q(f"SELECT COUNT(*) AS n FROM {t}")["n"][0] for t in tables}
overview = pl.DataFrame({"table": list(counts), "rows": list(counts.values())})
overview

In [ ]:
source_split = q(
    "SELECT source, COUNT(*) AS petitions FROM petition_source "
    "GROUP BY source ORDER BY petitions DESC"
)
source_split

## 2. Signatures and goal attainment

`signatures_total` is the raw count; `signatures_displayed` is the site's public-facing (rounded/inflated) figure.

In [ ]:
sig = q(
    "SELECT petition_id, signatures_total, signatures_displayed, goal, "
    "status FROM petition"
)
sig.select(
    n=pl.len(),
    total=pl.col("signatures_total").sum(),
    mean=pl.col("signatures_total").mean().round(0),
    median=pl.col("signatures_total").median(),
    p90=pl.col("signatures_total").quantile(0.90),
    maximum=pl.col("signatures_total").max(),
)

In [ ]:
vals = sig["signatures_total"].drop_nulls()
vals = vals.filter(vals > 0).to_numpy()
import numpy as np
bins = np.logspace(np.log10(vals.min()), np.log10(vals.max()), 30)
plt.figure()
plt.hist(vals, bins=bins)
plt.xscale("log")
plt.xlabel("signatures_total (log scale)")
plt.ylabel("petitions")
plt.title("Distribution of signatures")
plt.show()

In [ ]:
attain = sig.filter(pl.col("goal") > 0).with_columns(
    (pl.col("signatures_total") / pl.col("goal")).alias("attainment")
)
attain.select(
    petitions_with_goal=pl.len(),
    median_attainment=pl.col("attainment").median().round(2),
    share_reached_goal=(pl.col("attainment") >= 1.0).mean().round(3),
)

## 3. Engagement: signatures, comments, likes, displayed-vs-total

In [ ]:
eng = q(
    "SELECT petition_id, signatures_total, comment_total, "
    "signatures_displayed FROM petition"
)
corr = eng.select(pl.corr("signatures_total", "comment_total")).item()
print(f"corr(signatures, comments) = {corr:.3f}")
plt.figure()
plt.scatter(eng["signatures_total"].to_list(),
            eng["comment_total"].to_list(), s=12, alpha=0.5)
plt.xscale("log")
plt.yscale("symlog")
plt.xlabel("signatures_total")
plt.ylabel("comment_total")
plt.title("Signatures vs comments")
plt.show()

In [ ]:
gap = eng.filter(
    (pl.col("signatures_total") > 0)
    & pl.col("signatures_displayed").is_not_null()
).with_columns(
    (pl.col("signatures_displayed") / pl.col("signatures_total")).alias("ratio")
)
gap.select(
    median_displayed_over_total=pl.col("ratio").median().round(2),
    share_displayed_gt_total=(pl.col("ratio") > 1.0).mean().round(3),
)

In [ ]:
likes = q("SELECT likes FROM comment")
likes.select(
    mean=pl.col("likes").mean().round(2),
    median=pl.col("likes").median(),
    p99=pl.col("likes").quantile(0.99),
    maximum=pl.col("likes").max(),
)

## 4. Temporal: movement growth over time

In [ ]:
created = q(
    "SELECT substr(created_at, 1, 7) AS month, COUNT(*) AS petitions "
    "FROM petition WHERE created_at IS NOT NULL GROUP BY month ORDER BY month"
)
plt.figure(figsize=(11, 4))
plt.bar(created["month"].to_list(), created["petitions"].to_list())
plt.xticks(rotation=90)
plt.ylabel("petitions created")
plt.title("Petitions created per month")
plt.tight_layout()
plt.show()
created.tail(10)

In [ ]:
comments_month = q(
    "SELECT substr(created_at, 1, 7) AS month, COUNT(*) AS comments "
    "FROM comment WHERE created_at IS NOT NULL GROUP BY month ORDER BY month"
)
plt.figure(figsize=(11, 4))
plt.bar(comments_month["month"].to_list(),
        comments_month["comments"].to_list())
plt.xticks(rotation=90)
plt.ylabel("comments posted")
plt.title("Comments posted per month")
plt.tight_layout()
plt.show()

## 5. Geography

In [ ]:
dm_state = q(
    "SELECT state, COUNT(*) AS decision_makers FROM decision_maker "
    "WHERE state IS NOT NULL AND length(state) > 0 "
    "GROUP BY state ORDER BY decision_makers DESC"
)
top = dm_state.head(15)
plt.figure(figsize=(11, 4))
plt.bar(top["state"].to_list(), top["decision_makers"].to_list())
plt.ylabel("targeted decision makers")
plt.title("Top states by targeted decision makers")
plt.show()
dm_state.head(10)

In [ ]:
cities = q(
    "SELECT city, COUNT(*) AS comments FROM comment "
    "WHERE city IS NOT NULL AND length(city) > 0 "
    "GROUP BY city ORDER BY comments DESC"
)
cities.head(10)

## 6. Targets and campaign clusters

Recurring decision makers, and petitions that share targets (a proxy for coordinated campaigns).

In [ ]:
top_targets = q(
    "SELECT dm.display_name, dm.title, dm.type, dm.state, "
    "COUNT(DISTINCT pdm.petition_id) AS petitions FROM decision_maker dm "
    "JOIN petition_decision_maker pdm "
    "ON dm.decision_maker_id = pdm.decision_maker_id "
    "GROUP BY dm.decision_maker_id ORDER BY petitions DESC LIMIT 20"
)
top_targets.head(10)

In [ ]:
shared = q(
    "SELECT a.petition_id AS p1, b.petition_id AS p2, "
    "COUNT(*) AS shared_targets FROM petition_decision_maker a "
    "JOIN petition_decision_maker b "
    "ON a.decision_maker_id = b.decision_maker_id "
    "AND a.petition_id < b.petition_id "
    "GROUP BY p1, p2 HAVING shared_targets >= 2 "
    "ORDER BY shared_targets DESC LIMIT 20"
)
shared.head(10)

## 7. Tags and framing

In [ ]:
tags = q(
    "SELECT t.name, COUNT(*) AS petitions FROM tag t "
    "JOIN petition_tag pt ON t.tag_id = pt.tag_id "
    "GROUP BY t.tag_id ORDER BY petitions DESC LIMIT 20"
)
tags.head(10)

In [ ]:
tag_sig = q(
    "SELECT t.name, COUNT(DISTINCT pt.petition_id) AS petitions, "
    "CAST(AVG(p.signatures_total) AS INT) AS mean_signatures FROM tag t "
    "JOIN petition_tag pt ON t.tag_id = pt.tag_id "
    "JOIN petition p ON p.petition_id = pt.petition_id "
    "GROUP BY t.tag_id HAVING petitions >= 10 "
    "ORDER BY mean_signatures DESC LIMIT 15"
)
tag_sig.head(10)

## 8. Outcomes

In [ ]:
status = q(
    "SELECT status, COUNT(*) AS petitions, "
    "CAST(AVG(signatures_total) AS INT) AS mean_signatures, "
    "CAST(AVG(comment_total) AS INT) AS mean_comments "
    "FROM petition GROUP BY status ORDER BY petitions DESC"
)
status

## 9. Provenance comparison (DCDB vs change.org vs fullquery)

In [ ]:
by_source = q(
    "SELECT ps.source, COUNT(DISTINCT p.petition_id) AS petitions, "
    "CAST(AVG(p.signatures_total) AS INT) AS mean_signatures, "
    "CAST(AVG(p.comment_total) AS INT) AS mean_comments FROM petition_source ps "
    "JOIN petition p ON p.petition_id = ps.petition_id "
    "GROUP BY ps.source ORDER BY petitions DESC"
)
by_source

## Notes and caveats

- `organization` is empty for all petitions; use `creator_name` / decision makers instead.
- `is_verified_victory` is `False` for every row even where `status = VICTORY`; rely on `status`.
- Comments store no author id (only `city`), so repeat-commenter analysis is not possible.
- Provenance dedupe was by petition slug; alias slugs are reconciled to a single petition id on collection.